# CS 3892 / 5892 — HW4 · Interactive theorem proving in Lean 4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ttj/cs3892-examples/blob/main/notebooks/hw4-lean.ipynb)

**Out Tue Nov 3 · due Thu Nov 12, 11:59 p.m.** The handout is
[`assignments/hw4.md`](https://github.com/ttj/cs3892-fmstai-fall2026/blob/main/assignments/hw4.md) in the class
repository. This notebook installs Lean v4.33.0 and runs the checker on the starters as they ship:
every theorem is still a `sorry`. Nothing here is an answer.

**To work here:** open the folder icon (left) → `cs3892-examples/homework/hw4-lean/HW4/`, double-click a
`Part…lean` file, edit, save (Ctrl+S), and re-run the last cell. Colab has no Lean editor, so you will
not see goals as you type: for that, use VS Code with the Lean 4 extension, or the
[Lean web editor](https://live.lean-lang.org/) to experiment (it runs a different Lean version; the
checker here is what counts).
**A Colab runtime is deleted when it disconnects: download your files before you leave.**

## Setup

Run these two once.

In [1]:
# --- Setup: find the repo (clone on Colab), define helpers
import os, subprocess, sys, pathlib

REPO_URL = "https://github.com/ttj/cs3892-examples.git"
HW = "hw4-lean"

def _find_repo():
    here = pathlib.Path.cwd()
    for p in [here, *here.parents]:
        if (p / "homework" / HW).is_dir():
            return p
    dest = pathlib.Path("/content/cs3892-examples") if pathlib.Path("/content").is_dir() \
           else pathlib.Path.cwd() / "cs3892-examples"
    if not (dest / "homework" / HW).is_dir():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
    return dest

ROOT = _find_repo()
HD = ROOT / "homework" / HW
LD, PYD = HD / "HW4", HD / "python"
print("homework folder:", HD)

def show(path):
    path = pathlib.Path(path)
    print(f"--- {path.name} " + "-" * max(0, 60 - len(path.name)))
    print(path.read_text().rstrip())
    print()

def run(path, *args):
    """Run a Python script from its own folder and print its output."""
    path = pathlib.Path(path)
    r = subprocess.run([sys.executable, str(path), *map(str, args)], capture_output=True, text=True,
                       cwd=str(path.parent))
    print(r.stdout.rstrip())
    if r.returncode not in (0,):
        print(r.stderr.rstrip(), file=sys.stderr)
        raise RuntimeError(f"{path.name} exited with {r.returncode}")
    return r.stdout

def check(strict=False):
    """Build every part and report each theorem: proved / uses sorry / disallowed axiom / ..."""
    return run(HD / "check_hw4.py", *(["--strict"] if strict else []))

print("ready -- helpers: show(path), check(strict=False)")

homework folder: /content/cs3892-examples/homework/hw4-lean
ready -- helpers: show(path), check(strict=False)


In [2]:
# --- Lean v4.33.0. The first run installs elan (the Lean version manager) into ~/.elan and
# downloads Lean (about 575 MB, a minute or two on Colab); later runs reuse it.
out = check()
assert "HW4/Defs.lean    unchanged" in out and "Proved" in out

HW4 check — Lean v4.33.0   (Lean (version 4.33.0, x86_64-unknown-linux-gnu, commit d8b18978322de05a8f3dba51ef03cf5461676c17, Release))
  HW4/Defs.lean    unchanged
  lean-toolchain   unchanged

Part 1 — tactic ladder (15 points)
  uses sorry           rung1_compute           2   
  uses sorry           rung2_arith             2   
  uses sorry           rung3_and_intro         2   
  uses sorry           rung4_exists_elim       2   
  uses sorry           rung5_imp               2   
  uses sorry           rung6_induction         2   
  uses sorry           rung7_cases             3   

Part 2 — sets (20 points)
  uses sorry           subset_antisymm         4   
  uses sorry           demorgan_inter          4   
  uses sorry           union_distrib_inter     4   
  uses sorry           diff_union              4   
  uses sorry           subset_inter_iff        4   

Part 3 — the token bucket (30 points)
  uses sorry           P_not_inductive         6   
  uses sorry           Q_not_

## The given definitions

Transition systems, sets as predicates, and the token bucket. The checker compares this file with a
pinned SHA-256, so leave it as it is.

In [3]:
show(LD / "Defs.lean")

--- Defs.lean ---------------------------------------------------
/-
  HW4 — the given definitions. DO NOT EDIT THIS FILE.

  `check_hw4.py` compares it with a pinned SHA-256 and refuses to grade a copy
  that differs. Everything you prove lives in Part1Ladder.lean … Part4Parametric.lean.

  Core Lean 4 only (no Mathlib), Lean v4.33.0.
-/

namespace HW4

/-! ## Part 1's definitions -/

/-- Three times `n`, the long way. -/
def triple : Nat → Nat
  | 0 => 0
  | n + 1 => triple n + 3

inductive Light where
  | red | amber | green
deriving DecidableEq, Repr

/-- One phase of a traffic light. -/
def Light.next : Light → Light
  | .red => .green
  | .green => .amber
  | .amber => .red

/-! ## Transition systems

A transition system over a type of states: which states are initial, and which
pairs of states are steps. `Reachable` is the least set containing the initial
states and closed under steps -- an *inductive* predicate, so a fact about every
reachable state is proved by induction on ho

## Parts 1–4: the starters

Session 19 (Tue Nov 3) covers Parts 1 and 2; session 20 (Thu Nov 5) covers Parts 3 and 4.

In [ ]:
for f in ("Part1Ladder.lean", "Part2Sets.lean", "Part3Bucket.lean", "Part4Parametric.lean"):
    show(LD / f)

## What "proved" means here

`python/00_starter_checks.py` is the course's own test of the checker: on a scratch copy it closes one
rung with `native_decide` and another with a home-made `axiom`, and edits `Defs.lean`. The checker
has to reject all three. (A `sorry` compiles too, and proves nothing.)

In [ ]:
out = run(PYD / "00_starter_checks.py")
assert "native_decide -> disallowed axiom" in out and "exit 2" in out

## Your progress

Re-run this after every edit. `check(strict=True)` is the acceptance test: it fails unless every
theorem is proved. Part 4's written question and Part 5's log (`ai_log.md`) are graded by hand.

In [ ]:
out = check()
assert "Proved" in out

In [5]:
%cd /content/cs3892-examples/homework/hw4-lean
!python3 check_hw4.py --strict

/content/cs3892-examples/homework/hw4-lean
HW4 check — Lean v4.33.0   (Lean (version 4.33.0, x86_64-unknown-linux-gnu, commit d8b18978322de05a8f3dba51ef03cf5461676c17, Release))
  HW4/Defs.lean    unchanged
  lean-toolchain   unchanged

Part 1 — tactic ladder (15 points)
  proved               rung1_compute           2   no axioms
  proved               rung2_arith             2   propext, Quot.sound
  proved               rung3_and_intro         2   propext, Quot.sound
  proved               rung4_exists_elim       2   propext, Quot.sound
  proved               rung5_imp               2   no axioms
  proved               rung6_induction         2   propext, Quot.sound
  proved               rung7_cases             3   no axioms

Part 2 — sets (20 points)
  uses sorry           subset_antisymm         4   
  uses sorry           demorgan_inter          4   
  uses sorry           union_distrib_inter     4   
  uses sorry           diff_union              4   
  uses sorry           sub